# M7.5 · Quantization ve pruning

Modeli küçültmek. Üç sayıyı simetrik ve afin int8'e elle çeviriyor, büyüklüğe göre budamayı (magnitude pruning) elle yapıyor, saf Python'da quantize/dequantize ve budama fonksiyonlarını yazıyor, NumPy'da int32 birikimli int8 matmul ile tensör başına ve kanal başına ölçeklerin hata farkını ölçüyor, PyTorch'ta kendi int8 `QuantLinear` modülünü ve `torch.nn.utils.prune` ile global budamayı uyguluyor ve en sonda LLM'lerde kullanılan grup bazlı int4 ağırlık kuantizasyonunu, iki değeri bir bayta paketleyerek vektörel yazıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m7-5-quant-prune.html](https://m-gonul.github.io/pytorch-lab/m7-5-quant-prune.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m7_5_quant_prune.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def sym():\n        q, s = need(ns, \"quantize_sym\")([0.5, -1.2, 2.54])\n        ensure(list(q) == [25, -60, 127], f\"q = {q}\")\n        close(s, 0.02)\n        q, s = ns[\"quantize_sym\"]([0.0, 0.0])\n        ensure(list(q) == [0, 0], \"sıfır vektör\")\n        q, s = ns[\"quantize_sym\"]([1.0, -0.3], bits=4)\n        ensure(list(q) == [7, -2], f\"4 bit: {q}\")\n    R.test(\"quantize_sym\", sym)\n\n    def aff():\n        q, s, z = need(ns, \"quantize_affine\")([-1.0, 0.5, 3.0])\n        ensure(list(q) == [0, 96, 255] and z == 64, f\"q = {q}, z = {z}\")\n        close(s, 4 / 255)\n        q, s, z = ns[\"quantize_affine\"]([0.2, 0.9])\n        ensure(z == 0 and q[0] > 0, \"aralık 0'ı içerecek şekilde genişlemeli (min → 0)\")\n        close(ns[\"dequantize\"](*ns[\"quantize_affine\"]([0.0, 0.7, 1.4])), [0.0, 0.7, 1.4], atol=1.4 / 255)\n    R.test(\"quantize_affine (0 tam temsil edilir)\", aff)\n\n    def err():\n        rng = random.Random(0)\n        xs = [rng.gauss(0, 1) for _ in range(1000)]\n        q, s = ns[\"quantize_sym\"](xs)\n        d = need(ns, \"dequantize\")(q, s)\n        ensure(max(abs(a - b) for a, b in zip(xs, d)) <= s / 2 + 1e-12, \"hata ≤ s/2\")\n    R.test(\"yuvarlama hatası ≤ s/2\", err)\n\n    def pr():\n        ensure(list(need(ns, \"magnitude_prune\")([0.1, -2, 0.3, 1.5], 0.5)) == [0, -2, 0, 1.5], \"elle örnek\")\n        ensure(list(ns[\"magnitude_prune\"]([1, -1, 1, 2], 0.5)) == [0, 0, 1, 2], \"eşitlikte küçük indeks önce\")\n        w = [random.Random(1).uniform(-1, 1) for _ in range(100)]\n        ensure(sum(v == 0 for v in ns[\"magnitude_prune\"](w, 0.3)) == 30, \"%30 seyreklik\")\n    R.test(\"magnitude_prune\", pr)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    for f in (\"quant_per_tensor\", \"quant_per_channel\", \"int8_matmul\", \"rel_error\"):\n        R.test(f\"{f} döngüsüz\", (lambda f=f: max_loops(src, 0, func=f)))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    W = rng.normal(size=(64, 128)); W[3] *= 100\n    x = rng.normal(size=(16, 128))\n\n    def types():\n        q, s = need(ns, \"quant_per_channel\")(W)\n        ensure(q.dtype == np.int8 and np.shape(s) == (64,), \"int8 ve (out,) ölçek\")\n        ensure(np.abs(q).max(1).min() == 127, \"her kanalın en büyük değeri 127'ye oturmalı\")\n        q2, s2 = need(ns, \"quant_per_tensor\")(W)\n        ensure(q2.dtype == np.int8 and np.ndim(s2) == 0, \"int8 ve skaler ölçek\")\n    R.test(\"dtype ve ölçek şekilleri\", types)\n\n    def mm():\n        xq, sx = ns[\"quant_per_tensor\"](x)\n        wq, sw = ns[\"quant_per_channel\"](W)\n        want = (xq.astype(np.int64) @ wq.T.astype(np.int64)) * sx * sw\n        close(need(ns, \"int8_matmul\")(xq, sx, wq, sw), want)\n        big = np.full((1, 4096), 127, np.int8)\n        r = ns[\"int8_matmul\"](big, 1.0, big, 1.0)\n        ensure(r[0, 0] == 127 * 127 * 4096, \"int32 birikimi taşmamalı (int8/int16'da taşardı)\")\n    R.test(\"int8_matmul (int32 birikim)\", mm)\n\n    def cmp():\n        xq, sx = ns[\"quant_per_tensor\"](x)\n        ref = x @ W.T\n        e_t = need(ns, \"rel_error\")(ns[\"int8_matmul\"](xq, sx, *ns[\"quant_per_tensor\"](W)), ref)\n        e_c = ns[\"rel_error\"](ns[\"int8_matmul\"](xq, sx, *ns[\"quant_per_channel\"](W)), ref)\n        mask = np.ones(64, bool); mask[3] = False\n        e_t2 = ns[\"rel_error\"](ns[\"int8_matmul\"](xq, sx, *ns[\"quant_per_tensor\"](W))[:, mask], ref[:, mask])\n        e_c2 = ns[\"rel_error\"](ns[\"int8_matmul\"](xq, sx, *ns[\"quant_per_channel\"](W))[:, mask], ref[:, mask])\n        ensure(e_c2 < 0.2 * e_t2, \"aykırı olmayan kanallarda kanal başına ölçek çok daha iyi olmalı\")\n        R.table(\"Aykırı kanallı W (64 × 128), göreli hata\", [\"ölçek\", \"tüm çıktı\", \"aykırı kanal hariç\"],\n                [[\"tensör başına\", f\"{e_t:.2e}\", f\"{e_t2:.2e}\"], [\"kanal başına\", f\"{e_c:.2e}\", f\"{e_c2:.2e}\"]],\n                note=\"Tüm çıktı hatası aykırı kanal tarafından domine ediliyor; asıl kayıp diğer kanallarda.\")\n    R.test(\"tensör başına vs kanal başına\", cmp)\n",
 "d": "import copy\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    torch.manual_seed(0)\n\n    class Net(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.body = nn.Sequential(nn.Linear(128, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU())\n            self.head = nn.Linear(256, 10)\n\n        def forward(self, x):\n            return self.head(self.body(x))\n    m = Net()\n    x = torch.randn(64, 128)\n\n    def ql():\n        L = nn.Linear(16, 8)\n        Q = need(ns, \"QuantLinear\", \"sınıf\").from_float(L)\n        ensure(Q.weight_int8.dtype == torch.int8 and Q.scale.shape == (8,), \"int8 ağırlık, (out,) ölçek\")\n        ensure(int(Q.weight_int8.abs().amax(1).min()) == 127, \"kanal başına ölçek: her satırın max'ı 127\")\n        W = Q.weight_int8.float() * Q.scale[:, None]\n        ensure(((W - L.weight).abs() <= Q.scale[:, None] / 2 + 1e-7).all(), \"hata ≤ s/2\")\n    R.test(\"QuantLinear.from_float\", ql)\n\n    def qm():\n        q = need(ns, \"quantize_model\")(m)\n        ensure(not any(isinstance(mod, nn.Linear) for mod in q.modules()), \"iç içe tüm Linear'lar değişmeli\")\n        ensure(any(isinstance(mod, nn.Linear) for mod in m.modules()), \"orijinal model değişmemeli (deepcopy)\")\n        with torch.no_grad():\n            y, yq = m(x), q(x)\n        rel = ((yq - y).norm() / y.norm()).item()\n        ensure(rel < 0.02, f\"göreli hata {rel:.3e}\")\n        b0, b1 = need(ns, \"state_bytes\")(m), ns[\"state_bytes\"](q)\n        ensure(b1 < 0.3 * b0, f\"boyut {b0:,} → {b1:,}\")\n        return f\"göreli hata {rel:.2e}; state_dict {b0:,} → {b1:,} bayt ({b0 / b1:.1f}×)\"\n    R.test(\"quantize_model: doğruluk ve boyut\", qm)\n\n    def pr():\n        mm = copy.deepcopy(m)\n        need(ns, \"prune_global\")(mm, 0.6)\n        ws = [mod.weight for mod in mm.modules() if isinstance(mod, nn.Linear)]\n        tot = sum(w.numel() for w in ws); zeros = sum(int((w == 0).sum()) for w in ws)\n        ensure(abs(zeros / tot - 0.6) < 0.01, f\"seyreklik {zeros / tot:.3f}\")\n        ensure(not any(n.endswith(\"_orig\") for n, _ in mm.named_parameters()), \"prune.remove çağrılmalı (weight_orig kalmamalı)\")\n        per = [f\"{float((w == 0).float().mean()):.2f}\" for w in ws]\n        return f\"global %60: katman başına seyreklik {per} (global eşik katmanlara farklı oranlar düşürür)\"\n    R.test(\"prune_global\", pr)\n",
 "e": "import torch\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    for f in (\"quantize_int4\", \"pack_int4\", \"unpack_int4\", \"dequant_int4\"):\n        R.test(f\"{f} döngüsüz\", (lambda f=f: max_loops(src, 0, func=f)))\n    g = torch.Generator().manual_seed(0)\n    W = torch.randn(64, 256, generator=g)\n\n    def rt():\n        q = torch.randint(-8, 8, (5, 12), generator=g, dtype=torch.int8)\n        p = need(ns, \"pack_int4\")(q)\n        ensure(p.dtype == torch.uint8 and tuple(p.shape) == (5, 6), \"uint8, yarı genişlik\")\n        ensure(torch.equal(need(ns, \"unpack_int4\")(p), q), \"pack → unpack özdeşlik (negatifler dahil)\")\n        ensure(int(p[0, 0]) == ((int(q[0, 0]) & 0xF) | ((int(q[0, 1]) & 0xF) << 4)), \"düşük 4 bit çift indeks, yüksek 4 bit tek indeks\")\n    R.test(\"pack/unpack int4\", rt)\n\n    def qz():\n        q, s = need(ns, \"quantize_int4\")(W, 32)\n        ensure(q.dtype == torch.int8 and int(q.min()) >= -8 and int(q.max()) <= 7, \"q ∈ [-8, 7]\")\n        ensure(tuple(s.shape) == (64, 8), \"ölçek (out, in/group)\")\n        Wd = need(ns, \"dequant_int4\")(ns[\"pack_int4\"](q), s, 32)\n        err = (Wd - W).abs().reshape(64, 8, 32)\n        ensure((err <= s[..., None] / 2 + 1e-6).all(), \"grup içi hata ≤ s/2\")\n    R.test(\"quantize_int4 + dequant\", qz)\n\n    def table():\n        rows = []\n        Wo = W.clone(); Wo[:, 7] *= 30\n        for gsz in (256, 128, 32):\n            q, s = ns[\"quantize_int4\"](Wo, gsz)\n            Wd = ns[\"dequant_int4\"](ns[\"pack_int4\"](q), s, gsz)\n            rel = ((Wd - Wo).norm() / Wo.norm()).item()\n            bits = 4 + 16 / gsz\n            rows.append([str(gsz), f\"{rel:.3f}\", f\"{bits:.2f}\"])\n        rels = [float(r[1]) for r in rows]\n        ensure(rels[2] < rels[0], \"küçük grup daha az hata vermeli\")\n        R.table(\"int4, aykırı sütunlu W (64 × 256)\", [\"grup\", \"göreli hata\", \"bit/ağırlık (fp16 ölçek dahil)\"], rows,\n                note=\"Grup küçüldükçe aykırı değerin etkisi o gruba hapsolur; bedeli ölçekler için ek bit.\")\n    R.test(\"grup boyutu ve hata\", table)\n"
}

In [ ]:
import importlib, labcheck, checks_m7_5_quant_prune as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Simetrik kuantizasyon** ($b$ bit): $\ s=\max|x|/(2^{b-1}-1)$, $\ q=\operatorname{clamp}(\operatorname{round}(x/s))$, $\ \hat x=qs$.

**Afin** (zero-point'li, işaretsiz): $\ s=\frac{\max-\min}{2^b-1}$, $\ z=\operatorname{round}(-\min/s)$, $\ q=\operatorname{clamp}(\operatorname{round}(x/s)+z,\ 0,\ 2^b-1)$, $\ \hat x=(q-z)s$. Aralık sıfır etrafında simetrik değilse (ReLU sonrası aktivasyonlar) daha verimli.

**Ölçek tanecikliği:** tensör başına tek $s$; **kanal başına** her çıkış satırına bir $s$; **grup bazlı kuantizasyon** her satırı $g$ elemanlık gruplara bölüp her gruba bir $s$ (int4 LLM ağırlıklarında tipik $g=32\ldots128$).

**int8 matmul:** $\ y=x W^\top\approx(s_x s_w)\,(q_x q_W^\top)$; $q_xq_W^\top$ int32'de hesaplanır.

**Budama:** en küçük büyüklükteki ağırlıkları sıfırla. *Global* budama eşiği tüm katmanlar için tek; *katman başına* her katman aynı oranda.

## 1 · Elle

Simetrik ölçekte en büyük değer tam 127'ye oturur. Afin örnekte $\hat x=0.50196$, gerçek değer $0.5$: hata en fazla $s/2\approx0.0078$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Simetrik int8, $x=(0.5,\ -1.2,\ 2.54)$: $s=\max|x|/127$, $q=\operatorname{round}(x/s)$**

- `s`: $s$
- `q2`: $q_2$
- `q3`: $q_3$

**Afin uint8, aralık $[-1,\ 3]$: $s=(\max-\min)/255$, $z=\operatorname{round}(-\min/s)$**

- `zp`: Zero-point $z$
- `deq`: $x=0.5$ için $\hat x=(\operatorname{round}(x/s)+z-z)\cdot s$

**Büyüklüğe göre %50 budama: $w=(0.1,\ -2,\ 0.3,\ 1.5)$**

- `nz`: Sıfır olmayan ağırlık sayısı
- `w3`: Budama sonrası $w_3$

In [ ]:
cevaplar = dict(
    s=None,
    q2=None,
    q3=None,
    zp=None,
    deq=None,
    nz=None,
    w3=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"s":[0.02,0.00001],"q2":[-60,0],"q3":[127,0],"zp":[64,0],"deq":[0.501961,0.00001],"nz":[2,0],"w3":[0,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `quantize_sym(xs, bits=8)` → `(q, s)`; tümü sıfırsa `s = 1`.
- `quantize_affine(xs, bits=8)` → `(q, s, z)`; $\min$'i $\le0$, $\max$'ı $\ge0$ olacak şekilde genişlet (0 tam temsil edilsin), sonra kırp.
- `dequantize(q, s, z=0)`.
- `magnitude_prune(ws, sparsity)` — $k=\lfloor\text{sparsity}\cdot n\rfloor$ en küçük $|w|$'yi sıfırla (eşitlikte küçük indeks önce budanır).

In [ ]:
%%writefile stage_b.py
import math


def quantize_sym(xs, bits=8):
    ...


def quantize_affine(xs, bits=8):
    ...


def dequantize(q, s, z=0):
    ...


def magnitude_prune(ws, sparsity):
    ...


print(quantize_sym([0.5, -1.2, 2.54]))           # ([25, -60, 127], 0.02)
print(quantize_affine([-1.0, 0.5, 3.0]))         # ([0, 96, 255], 0.01569, 64)
print(magnitude_prune([0.1, -2, 0.3, 1.5], 0.5))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**. `W: (out, in)`, `x: (B, in)`.

- `quant_per_tensor(W)` → `(q int8, s skaler)`; `quant_per_channel(W)` → `(q int8, s (out,))`.
- `int8_matmul(xq, sx, wq, sw)` → `(xq.astype(int32) @ wq.T.astype(int32)) * sx * sw` (`sw` skaler ya da `(out,)`).
- `rel_error(y, y_ref)` → $\lVert y-y_{ref}\rVert/\lVert y_{ref}\rVert$.

In [ ]:
%%writefile stage_c.py
import numpy as np


def quant_per_tensor(W):
    ...


def quant_per_channel(W):
    ...


def int8_matmul(xq, sx, wq, sw):
    ...


def rel_error(y, y_ref):
    ...


rng = np.random.default_rng(0)
W = rng.normal(size=(64, 128)); W[3] *= 100       # aykırı kanal
x = rng.normal(size=(16, 128))
xq, sx = quant_per_tensor(x)
for name, (wq, sw) in [("tensör", quant_per_tensor(W)), ("kanal", quant_per_channel(W))]:
    print(name, rel_error(int8_matmul(xq, sx, wq, sw), x @ W.T))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `QuantLinear` — `from_float(linear)` sınıf metodu: kanal başına simetrik int8; `weight_int8` (int8) ve `scale` (fp32, `(out,)`) **buffer** olarak, `bias` kopyası. `forward(x)`: `F.linear(x, weight_int8.float() * scale[:, None], bias)` (ağırlık-yalnız kuantizasyon: depolama int8, hesap fp32).
- `quantize_model(model)` — tüm `nn.Linear`'ları (iç içe dahil) `QuantLinear` ile değiştir; yeni model döndür (`copy.deepcopy`).
- `state_bytes(model)` — `state_dict` tensörlerinin bayt toplamı.
- `prune_global(model, amount)` — tüm `nn.Linear` ağırlıklarında `prune.global_unstructured(…, L1Unstructured, amount)`, sonra `prune.remove` (maskeyi kalıcı yap).

Not: `torch.ao.quantization` PyTorch'ta kullanımdan kaldırılıyor; yerini `torchao` alıyor. Burada mekanizmayı görmek için kendi modülünü yazıyorsun.

In [ ]:
%%writefile stage_d.py
import copy
import torch
import torch.nn.functional as F
import torch.nn.utils.prune as prune
from torch import nn


class QuantLinear(nn.Module):
    @classmethod
    def from_float(cls, linear):
        ...

    def forward(self, x):
        ...


def quantize_model(model):
    ...


def state_bytes(model):
    ...


def prune_global(model, amount):
    ...


torch.manual_seed(0)
m = nn.Sequential(nn.Linear(128, 256), nn.ReLU(), nn.Linear(256, 10))
q = quantize_model(m)
x = torch.randn(4, 128)
print((q(x) - m(x)).abs().max(), state_bytes(m), state_bytes(q))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

LLM tarzı ağırlık-yalnız **int4**, grup bazlı kuantizasyon, **döngü yok**:

- `quantize_int4(W, group)` → `(q, s)`: `W (out, in)`, `in` `group`'a bölünür; `W.view(out, in/group, group)`, grup başına $s=\max|w|/7$, $q\in[-8,7]$ (int8 tensörde).
- `pack_int4(q)` → `uint8 (out, in/2)`: iki değeri bir bayta: `(q[..., 0::2] & 0xF) | ((q[..., 1::2] & 0xF) << 4)`.
- `unpack_int4(p)` → `int8`; 4 bitlik iki tümleyeni geri işaretle (`v - 16 if v > 7`).
- `dequant_int4(p, s, group)` → fp32 `W`.

Kontrol yeniden yapılandırma hatasını grup boyutuna göre ve bit/ağırlık maliyetini (ölçekler dahil) tablolar.

In [ ]:
%%writefile stage_e.py
import torch


def quantize_int4(W, group):
    """→ (q int8 [-8, 7], s (out, in/group)). Döngü yok."""
    ...


def pack_int4(q):
    ...


def unpack_int4(p):
    ...


def dequant_int4(p, s, group):
    ...


torch.manual_seed(0)
W = torch.randn(64, 256)
q, s = quantize_int4(W, 32)
p = pack_int4(q)
print(p.dtype, p.shape, (dequant_int4(p, s, 32) - W).abs().max())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Kaynaklar

- [Jacob ve ark. — Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference](https://arxiv.org/abs/1712.05877) — arXiv
- [Dettmers ve ark. — LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale](https://arxiv.org/abs/2208.07339) — arXiv
- [Frantar ve ark. — GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers](https://arxiv.org/abs/2210.17323) — arXiv
- [Han, Mao, Dally — Deep Compression: Pruning, Trained Quantization and Huffman Coding](https://arxiv.org/abs/1510.00149) — arXiv
- [PyTorch Tutorials — Pruning Tutorial](https://docs.pytorch.org/tutorials/intermediate/pruning_tutorial.html) — BSD-3-Clause
- [torchao — PyTorch native quantization and sparsity](https://github.com/pytorch/ao) — BSD-3-Clause